## ChemSimGuide - Interactive Agent Runner
This notebook initializes all components of the ChemSimGuide agent
(configuration, data, embedding, tools, graph) and runs the
interactive chat session.

**Prerequisites:**
1.  Ensure you have created a `.env` file in the project root directory
    containing your `GOOGLE_API_KEY`.
2.  Ensure your Cantera documentation `.txt` files are placed inside
    the `../data/cantera_docs/` directory relative to this notebook.3.  Install all required libraries: `pip install -r ../requirements.txt`
4.  Ensure your Python environment (e.g., conda env 'CSG') is activated.



# 1. Installs (if needed)

In [1]:
#Uncomment the line below if you haven't installed requirements via requirements.txt
#!pip install -q -U google-generativeai chromadb langchain-google-genai langchain-core langgraph langgraph-prebuilt numpy python-dotenv ipython

# 2. Imports

In [ ]:
from google import genai
from google.genai import types
from google.api_core import retry
from google.genai import types

import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage 
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage, ToolMessage, BaseMessage
from langgraph.prebuilt import ToolNode

from typing import Annotated, List, Dict, Any, Optional, TypedDict, Literal
from typing_extensions import TypedDict 

from chemsimguide.config import (
        GOOGLE_API_KEY, LLM_MODEL_NAME, EMBEDDING_MODEL_NAME,
        DB_NAME, WELCOME_MSG, CHEMSIM_SYSINT, FULL_SCRIPT_GENERATION_PROMPT,
        N_RAG_RESULTS_DEFAULT, DB_PATH
    )
from chemsimguide.data_handler import load_and_chunk_data
from chemsimguide.embedding import *
from chemsimguide.tools import *


import os
import re
import io
import time


from IPython.display import HTML, Markdown, display,clear_output,Image
from pprint import pprint


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

Loaded environment variables from: c:\Users\souvi\OneDrive - The University of Western Ontario\RESEARCH\Code\My Repos\ChemSimGuide\.env


In [2]:
client = genai.Client(api_key=GOOGLE_API_KEY)

# 3. Configuration and Initialization

In [3]:
# --- Initialize Embedding Function ---
embed_fn = GeminiEmbeddingFunction(client, document_mode=True)

# --- Load the Persistent DB ---
db_path = DB_PATH
print(f"Initializing persistent ChromaDB client at: {db_path}")
chroma_client = chromadb.PersistentClient(path=db_path)
print(f"Getting or creating collection: {DB_NAME}")
db = chroma_client.get_or_create_collection(
    name=DB_NAME,
    embedding_function=embed_fn 
)
print(f"Accessed collection '{DB_NAME}'. Current document count: {db.count()}")

Initializing persistent ChromaDB client at: data/vector_db/cantera
Getting or creating collection: CanteraDocs
Accessed collection 'CanteraDocs'. Current document count: 59505


### --- Test ChromaDB Queries ---

In [5]:
import random, statistics, time
from pprint import pprint

# ── Config ──────────────────────────────────────────────────────────────
COLLECTION   = db          # or test_db
N_RUNS       = 25          # how many repeated queries
N_RESULTS    = 10          # top-k to fetch
QUERY_SET    = [
    "How to define a gas mixture in Cantera?",
    "Arrhenius equation units",
    "Surface reaction rate example",
    "Constant-volume reactor script",
    "Premixed flame speed tutorial",
]

print(f"Timing RAG on collection “{COLLECTION.name}” (docs = {COLLECTION.count()})")
# ── Benchmark loop ───────────────────────────────────────────────────────
times = []
for i in range(N_RUNS):
    q = random.choice(QUERY_SET)
    t0 = time.perf_counter()
    _ = COLLECTION.query(
        query_texts=[q],
        n_results=N_RESULTS,
    )
    times.append(time.perf_counter() - t0)

# ── Stats ────────────────────────────────────────────────────────────────
avg   = sum(times) / len(times)
stdev = statistics.stdev(times)
conf95 = 1.96 * stdev / (len(times) ** 0.5)

print(f"\nAvg latency: {avg*1000:.1f} ms ± {conf95*1000:.1f} ms  (95 % CI, N={N_RUNS})")
print(f"   min: {min(times)*1000:.1f} ms")
print(f"   max: {max(times)*1000:.1f} ms")


Timing RAG on collection “CanteraDocs” (docs = 59505)

Avg latency: 192.9 ms ± 23.8 ms  (95 % CI, N=25)
   min: 169.6 ms
   max: 481.4 ms


### --Test RAG--

In [ ]:
system_prompt = """
You are ChemSimGuide, a helpful and knowledgeable AI assistant specializing in setting up simulations using the Cantera Python library.

Your primary goal is to interact with the RAG Database and retrieve knowledge about the query
"""
# Switch to query mode when generating embeddings.
embed_fn.document_mode = False

# Search the Chroma DB using the specified query.
query = "I want to do a Kinetic Simulation of Methanol Synthesis using Cantera"

result = db.query(query_texts=[query], n_results=5)
[all_passages] = result["documents"]

prompt = f"{system_prompt} QUESTION: {query}"

# Add the retrieved documents to the prompt.
for passage in all_passages:
    passage_oneline = passage.replace("\n", " ")
    prompt += f"PASSAGE: {passage_oneline}\n"

answer = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt)

Markdown(answer.text)

In [ ]:
import time
from pprint import pprint
from IPython.display import Markdown

# -------------------------------------------------
# 1. Build the prompt with RAG retrieval
# -------------------------------------------------
system_prompt = """
You are ChemSimGuide, a helpful and knowledgeable AI assistant
specializing in setting up simulations using the Cantera Python library.
Your primary goal is to interact with the RAG database and retrieve
knowledge about the query.
"""

query = "I want to do a Kinetic Simulation of Methanol Synthesis using Cantera"

# -- a) RAG timer ---------------------------------------------------------
embed_fn.document_mode = False                # query mode
t_rag0 = time.perf_counter()
result = db.query(query_texts=[query], n_results=5)
rag_time = time.perf_counter() - t_rag0
print(f"🔍  RAG query time: {rag_time*1000:.1f} ms")

[all_passages] = result["documents"]

# Build prompt
prompt = f"{system_prompt}\nQUESTION: {query}\n"
for passage in all_passages:
    prompt += f"PASSAGE: {passage.replace(chr(10), ' ')}\n"

# -------------------------------------------------
# 2. Call Gemini and time the response
# -------------------------------------------------
t_llm0 = time.perf_counter()
answer = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt,
    
)
llm_time = time.perf_counter() - t_llm0
print(f"🧠  LLM generation time: {llm_time:.2f} s")

# -------------------------------------------------
# 3. Display answer (optional)
# -------------------------------------------------
Markdown(answer.text)


## Defining the RAG Search Tool

In [4]:
from chemsimguide import tools as csg_tools

# Inject the handles ONCE, right after you build db & embed_fn:
csg_tools.db = db
csg_tools.embed_fn = embed_fn

# Simple ad-hoc testing (n_results as normal keyword):
#print(csg_tools.search_cantera_docs("constant-pressure reactor", n_results=3))

### 4.1. Defining the Agent State

In [5]:
from chemsimguide.state import ChemSimState, GuidanceStep
from chemsimguide import tools as csg_tools

### 4.3 Defining the Cantera Code Generation Tool

In [6]:
# Inject shared objects once
csg_tools.db = db
csg_tools.embed_fn = embed_fn
csg_tools.llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash',google_api_key=GOOGLE_API_KEY)
# Grab the Tool object for the agent
generate_cantera_code_tool = csg_tools.generate_cantera_code

In [ ]:
## Testing the tool directly
result = csg_tools.generate_cantera_code.invoke(
    {"simulation_goal": "simulate ignition delay of H2/O2 at 1000 K","n_rag_results": 5,})
print(result[:500])

In [7]:
from chemsimguide import nodes as csg_nodes
from chemsimguide.routing import *

### 4.5 Assembling and Compiling the Graph

In [8]:
llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash',google_api_key=GOOGLE_API_KEY)
tools = [ search_cantera_docs, generate_cantera_code ]
llm_with_tools = llm.bind_tools(tools)

In [9]:
from chemsimguide.graph import build_chem_sim_graph
graph = build_chem_sim_graph(llm_with_tools)

Building ChemSimGuide LangGraph …
ChemSimGuide graph compiled successfully.


# 5. Interactive Demonstration

In [ ]:
# ── 2. Initialise state ────────────────────────────────────────────────
state = {"messages": []}          # empty conversation

# ── 3. REPL loop ───────────────────────────────────────────────────────
print("ChemSimGuide chat — type 'quit' to exit\n")

while True:
    # Run the graph → it will ask the user its first question
    state = graph.invoke(state)

    # Check if graph set finished flag (user typed quit from human_node)
    if state.get("finished_guidance"):
        print("Session ended.")
        break

    # Extract the most recent AI message so we know what the assistant said
    last_ai = state["messages"][-1]
    if hasattr(last_ai, "content"):
        print(f"\nChemSimGuide: {last_ai.content}")

    # Get user reply
    user_input = input("\nYou: ").strip()
    if user_input.lower() in {"quit", "exit", "bye", "q"}:
        # Signal exit on next pass
        state["finished_guidance"] = True
    else:
        # Append HumanMessage for the next graph turn
        from langchain_core.messages import HumanMessage
        state["messages"].append(HumanMessage(content=user_input))